# Reviewer entry point: reverse transfer

Use the environment in Notebook 01. Set `MAH_PATH` to the frozen release and
`OLI_PATH` or `OLI_ZIP` to the official OLI2MSI data inside the working directory.
Use the original `PREFLIGHT_PATH` or `BENCHMARK_ZIP` to reproduce the model panel.
Set `RUN_TRAINING=False` to reuse the original OLI2MSI-trained run directory,
including its checkpoints and receipts. Select checkpoints on validation before
setting `RUN_TEST=True` and running the locked test section.


# Notebook 4: OLI2MSI-trained published baselines (compact DGX)

**No NAS run, checkpoint, source checkout or prior NAS notebook is required.**
Train only RGB models passing your original Mahimandal preflight. Select weights
with OLI2MSI validation and evaluate on its 100 official test pairs and the frozen
731-pair Mahimandal test set. RGB only; OLI2MSI supplies no verified NIR.

Full **160×160 -> 480×480** frames for train, validation and test; 50-epoch ceiling,
patience 3, batch 1, Adam 1e-4. Original ID-based OLI split and metric rules are
unchanged. Existing matching Notebook 4 checkpoints resume at the same result
path. Nothing is deleted, and no other student's directory is accessed.

Set relative paths in cell 1. Existing OLI imagery is reused; do not download it
again. Point PREFLIGHT_PATH at a restored preflight_status.json if needed, or
supply the already uploaded original benchmark ZIP. Only model source code is
cloned if missing. Use the working additional-seeds DGX Python kernel. Rasterio
and PyYAML may be installed privately if absent; no global changes or torch/numpy
upgrades. GPU discovery accepts an unset CUDA_VISIBLE_DEVICES when PyTorch sees
exactly one assigned device.

Run setup, dataset/source audit and training. Leave RUN_TEST=False until choices
are frozen; then enable it and run only evaluation and report/export. No NAS
architectures enter this dataset benchmark. SRVGG uses the preserved paired
perceptual/GAN warmup policy; an early stop before warmup is not a trained GAN.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, subprocess, zipfile, stat, hashlib
from IPython.display import display, FileLink

# Relative to your own 25mcsa19 directory; edit only these paths/settings.
MAH_PATH = "mahimandal160_25mcsa19/datasets/mahimandal160/reflectance16"
OLI_PATH = ""       # Existing folder containing train_lr/train_hr/test_lr/test_hr.
OLI_ZIP = "./OLI2MSI-dataset.zip"        # Optional already-uploaded OLI2MSI ZIP, if not extracted.
PREFLIGHT_PATH = "" # Optional preflight_status.json from the original Mahimandal run.
BENCHMARK_ZIP = "mahimandal160_paper_benchmark_v1_complete_compact_20261002T113656696649Z.zip"
SOURCE_PATH = ""    # Optional existing pinned benchmark source folder.
DOWNLOAD_OLI = False # Enable only if the dataset is absent and Kaggle CLI is installed.
PYTHON = sys.executable
DEVICE = "cuda"
EPOCHS, PATIENCE, BATCH_SIZE, WORKERS, SEED = 50, 3, 1, 2, 42
LEARNING_RATE = 1e-4
RUN_TRAINING = True
RUN_TEST = False    # Enable after validation checkpoint selection is finalized.

CWD = Path.cwd().resolve()
SESSION = next((p for p in (CWD,) + tuple(CWD.parents) if p.name.casefold() == '25mcsa19'), None)
if SESSION is None:
    raise RuntimeError('Start inside your own 25mcsa19 folder; no shared directory is searched')
def owned(path):
    result = (SESSION / path).resolve()
    try: result.relative_to(SESSION)
    except ValueError: raise RuntimeError('Refusing access outside your folder: ' + str(result))
    return result

BENCH_HOME = owned('mahimandal160_25mcsa19')
CROSS_HOME = owned('mahimandal_oli_transfer_25mcsa19')
CODE_DIR = owned(CROSS_HOME / 'code_notebook4_compact')
RESULTS = owned(CROSS_HOME / 'results/notebook4') # Preserve old checkpoint layout.
LOCAL_SITE = owned(CROSS_HOME / '.python_packages/notebook4')
for p in (CODE_DIR, RESULTS, LOCAL_SITE, CROSS_HOME / '.cache', CROSS_HOME / 'tmp'):
    owned(p).mkdir(parents=True, exist_ok=True)
ENV = os.environ.copy()
ENV.update(PYTHONDONTWRITEBYTECODE='1', PYTHONUNBUFFERED='1',
           XDG_CACHE_HOME=str(CROSS_HOME / '.cache'), PIP_CACHE_DIR=str(CROSS_HOME / '.cache/pip'),
           TORCH_HOME=str(CROSS_HOME / '.cache/torch'), MPLCONFIGDIR=str(CROSS_HOME / '.cache/matplotlib'),
           TMPDIR=str(CROSS_HOME / 'tmp'), TMP=str(CROSS_HOME / 'tmp'), TEMP=str(CROSS_HOME / 'tmp'))
sites = [LOCAL_SITE, owned('mahimandal160_25mcsa19/.python-packages/site-packages')]
ENV['PYTHONPATH'] = os.pathsep.join(str(p) for p in [CODE_DIR] + sites)
def run(command, capture=False):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    return subprocess.run(command, env=ENV, cwd=str(CROSS_HOME), check=True,
                          stdout=subprocess.PIPE if capture else None, text=True)
def safe_extract(archive, destination):
    archive, destination = owned(archive), owned(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            target = owned(destination / member.filename)
            target.relative_to(destination)
            if stat.S_ISLNK(member.external_attr >> 16):
                raise RuntimeError('Symlink in ZIP: ' + member.filename)
            if target.exists() and not member.is_dir():
                raise FileExistsError('Partial extraction exists; choose another destination, preserving: ' + str(target))
        z.extractall(destination)

# Reuse DGX PyTorch; install only missing lightweight/geospatial packages privately.
for module, requirements in (
    ('rasterio', ['rasterio>=1.3,<1.4','affine','attrs','certifi','click<8.2','click-plugins','cligj','snuggs','pyparsing<3.2']),
    ('yaml', ['PyYAML>=6,<7']),
):
    check = subprocess.run([PYTHON, '-c', 'import ' + module], env=ENV, cwd=str(CROSS_HOME))
    if check.returncode:
        run([PYTHON, '-m', 'pip', 'install', '--target', LOCAL_SITE, '--no-deps',
             '--only-binary=:all:', *requirements])
probe = """import sys, os, torch, numpy, pandas, rasterio, matplotlib, yaml
from pathlib import Path
print('Python:', sys.executable, 'PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', '<unset>'))
for path in sys.argv[2:]:
    try: Path(torch.__file__).resolve().relative_to(Path(path).resolve())
    except ValueError: pass
    else: raise RuntimeError('Workspace-local duplicate PyTorch detected; select the working DGX environment')
if sys.argv[1] == 'cuda':
    print('Visible GPUs:', torch.cuda.device_count())
    if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
        raise RuntimeError('Exactly one administrator/scheduler-visible GPU is required; no physical GPU is selected')
    print('GPU:', torch.cuda.get_device_name(0))
"""
run([PYTHON, '-c', probe, DEVICE, *sites])
print('Notebook 4 has no dependency on any NAS run or NAS checkpoint.')


In [ ]:
MAH_ROOT = owned(MAH_PATH)
manifests = [p for p in (MAH_ROOT / 'paper_release_manifest.csv', MAH_ROOT / 'paper_reports/paper_release_manifest.csv') if p.is_file()]
if len(manifests) != 1:
    raise FileNotFoundError('Set MAH_PATH to the frozen reflectance16 release root: ' + str(MAH_ROOT))

if PREFLIGHT_PATH:
    PREFLIGHT = owned(PREFLIGHT_PATH)
else:
    bases = [BENCH_HOME / 'outputs/mahimandal160_paper_benchmark_v1',
             BENCH_HOME / 'restored_seed42', owned('restored_benchmark_results'),
             CROSS_HOME / 'restored_benchmark']
    found = sorted({owned(p) for base in bases for p in owned(base).rglob('preflight_status.json')})
    if not found:
        archive = owned(BENCHMARK_ZIP)
        if not archive.is_file():
            raise FileNotFoundError('Set PREFLIGHT_PATH to the restored original preflight_status.json, or upload the benchmark result ZIP: ' + str(archive))
        restored = owned(CROSS_HOME / 'restored_benchmark')
        found = sorted(restored.rglob('preflight_status.json'))
        if not found:
            safe_extract(archive, restored)
            found = sorted(restored.rglob('preflight_status.json'))
    if len(found) != 1:
        raise RuntimeError('Set PREFLIGHT_PATH explicitly; candidates: ' + str(found))
    PREFLIGHT = owned(found[0])
if not PREFLIGHT.is_file(): raise FileNotFoundError(PREFLIGHT)

OLI_CONTAINER = owned(CROSS_HOME / 'datasets/oli2msi')
def oli_roots(container):
    names = {'trainlr','trainhr','testlr','testhr'}
    found = []
    if not container.is_dir(): return found
    candidates = [container] + [p.parent for p in container.rglob('*')
                                if p.is_dir() and ''.join(c for c in p.name.casefold() if c.isalnum()) == 'trainlr']
    for candidate in candidates:
        owned(candidate)
        children = {''.join(c for c in p.name.casefold() if c.isalnum()) for p in candidate.iterdir() if p.is_dir()}
        if names.issubset(children): found.append(candidate.resolve())
    return sorted(set(found))

if OLI_PATH:
    roots = oli_roots(owned(OLI_PATH))
else:
    roots = sorted(set(oli_roots(OLI_CONTAINER) + oli_roots(owned('datasets/oli2msi'))))
if not roots:
    archive = owned(OLI_ZIP) if OLI_ZIP else OLI_CONTAINER / 'oli2msi-thesis.zip'
    if not archive.is_file() and DOWNLOAD_OLI:
        credentials = owned(BENCH_HOME / 'secrets/kaggle/kaggle.json')
        if not credentials.is_file():
            raise FileNotFoundError('Place your own kaggle.json here, or set OLI_PATH/OLI_ZIP: ' + str(credentials))
        ENV['KAGGLE_CONFIG_DIR'] = str(credentials.parent)
        OLI_CONTAINER.mkdir(parents=True, exist_ok=True)
        run([PYTHON, '-c', 'from kaggle.cli import main; main()', 'datasets', 'download',
             '-d', 'twilight2002/oli2msi-thesis', '-p', OLI_CONTAINER])
    if not archive.is_file():
        raise FileNotFoundError('Dataset already downloaded? Set OLI_PATH to its extracted root or OLI_ZIP to your uploaded ZIP. No download was attempted.')
    safe_extract(archive, OLI_CONTAINER / 'extracted_compact')
    roots = oli_roots(OLI_CONTAINER)
if len(roots) != 1: raise RuntimeError('Set OLI_PATH explicitly; candidates: ' + str(roots))
OLI_ROOT = owned(roots[0])
print('Mahimandal:', MAH_ROOT)
print('OLI2MSI:', OLI_ROOT)
print('Original preflight:', PREFLIGHT)


In [ ]:
PAYLOAD = {'adapters.py': '"""Pinned upstream x3 model adapters with explicit modality capability checks."""\nfrom __future__ import annotations\n\nimport contextlib\nimport importlib.util\nimport io\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport types\nfrom types import SimpleNamespace\n\nimport torch\nfrom torch import nn\nimport yaml\n\n\nREGISTRY = json.loads(Path(__file__).with_name("registry.json").read_text(encoding="utf-8"))\n\n\nclass _Registry:\n    def register(self, obj=None, **_):\n        def decorator(value):\n            return value\n        return decorator(obj) if obj is not None else decorator\n\n\ndef _install_basicsr_stubs() -> None:\n    """Provide only the two BasicSR interfaces used by pinned architecture files."""\n    if "basicsr.utils.registry" in sys.modules:\n        return\n    basicsr = types.ModuleType("basicsr")\n    basicsr.__path__ = []\n    utils = types.ModuleType("basicsr.utils")\n    utils.__path__ = []\n    registry = types.ModuleType("basicsr.utils.registry")\n    registry.ARCH_REGISTRY = _Registry()\n    archs = types.ModuleType("basicsr.archs")\n    archs.__path__ = []\n    arch_util = types.ModuleType("basicsr.archs.arch_util")\n    arch_util.to_2tuple = lambda value: value if isinstance(value, tuple) else (value, value)\n    arch_util.trunc_normal_ = torch.nn.init.trunc_normal_\n    sys.modules.update({"basicsr": basicsr, "basicsr.utils": utils,\n                        "basicsr.utils.registry": registry, "basicsr.archs": archs,\n                        "basicsr.archs.arch_util": arch_util})\n\n\ndef git(root: str | Path, *arguments: str) -> str:\n    return subprocess.check_output(["git", "-C", str(root), *arguments], text=True).strip()\n\n\ndef verify_source(root: str | Path, spec: dict) -> None:\n    root = Path(root)\n    if git(root, "rev-parse", "HEAD") != spec["revision"]:\n        raise ValueError(f"Wrong source revision: {root}; use a separate sources directory")\n    if git(root, "status", "--porcelain", "--untracked-files=no"):\n        raise ValueError(f"Tracked upstream files were modified: {root}")\n\n\ndef fetch(sources: str | Path) -> None:\n    sources = Path(sources)\n    sources.mkdir(parents=True, exist_ok=True)\n    unique = {spec["source_dir"]: spec for spec in REGISTRY.values()}\n    for spec in unique.values():\n        root = sources / spec["source_dir"]\n        if not root.exists():\n            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",\n                            f"https://github.com/{spec[\'repo\']}.git", str(root)], check=True)\n            subprocess.run(["git", "-C", str(root), "sparse-checkout", "set", "--no-cone",\n                            "*.py", "*.yaml", "*.yml", "*.md", "*LICENSE*"], check=True)\n            subprocess.run(["git", "-C", str(root), "checkout", spec["revision"]], check=True)\n        verify_source(root, spec)\n        print(spec["source_dir"], spec["revision"], flush=True)\n\n\ndef _load_file(name: str, path: Path):\n    module_spec = importlib.util.spec_from_file_location(name, path)\n    if module_spec is None or module_spec.loader is None:\n        raise ImportError(f"Cannot import {name} from {path}")\n    module = importlib.util.module_from_spec(module_spec)\n    sys.modules[name] = module\n    module_spec.loader.exec_module(module)\n    return module\n\n\ndef _clear_upstream_modules() -> None:\n    for key in list(sys.modules):\n        if key.split(".")[0] in {"model", "models", "ops", "blocks", "components"}:\n            del sys.modules[key]\n\n\ndef _classic_model(name: str, module, channels: int):\n    args = dict(scale=[3], n_colors=channels, n_feats=64, rgb_range=255,\n                res_scale=1, n_resgroups=10, n_resblocks=20, reduction=16,\n                n_basic_modules=10, G0=64, RDNkSize=3, RDNconfig="B",\n                no_upsampling=False)\n    if name == "EDSR":\n        args.update(n_feats=256, n_resblocks=32, res_scale=0.1)\n    if name == "SAN":\n        args.update(n_resgroups=20, n_resblocks=10)\n    return module.make_model(SimpleNamespace(**args)), args\n\n\ndef load(name: str, sources: str | Path, modality: str):\n    if modality not in {"rgb", "rgbnir"}:\n        raise ValueError(modality)\n    channels = 3 if modality == "rgb" else 4\n    if name == "bicubic":\n        return None, {"name": "bicubic", "scale": 3, "input_channels": channels,\n                      "output_channels": channels, "parameters": 0,\n                      "provenance": "PyTorch bicubic interpolation; align_corners=False",\n                      "suite": "baseline", "range_multiplier": 1, "range_offset": 0}\n    if name not in REGISTRY:\n        raise KeyError(name)\n    spec = dict(REGISTRY[name])\n    if modality not in spec["modalities"]:\n        raise ValueError(f"{name} does not have a verified {modality} construction; supported={spec[\'modalities\']}")\n    root = (Path(sources) / spec["source_dir"]).resolve()\n    verify_source(root, spec)\n    entry = root / spec["entry"]\n    _clear_upstream_modules()\n    if name in {"DCM", "HSENet", "HAUNet", "RCAN", "SAN", "RDN", "EDSR"}:\n        import_root = entry.parent.parent\n    else:\n        import_root = root\n    sys.path.insert(0, str(import_root))\n    try:\n        with contextlib.redirect_stdout(io.StringIO()):\n            if name == "HAT":\n                _install_basicsr_stubs()\n            if name == "RealESRGAN_SRVGG_X3":\n                _install_basicsr_stubs()\n            module = _load_file("upstream_" + name, entry)\n            if name == "CAFRN":\n                kwargs = dict(in_channels=channels, out_channels=channels, num_features=64,\n                              num_steps=7, reduction=16, upscale_factor=3,\n                              act_type="prelu", norm_type=None)\n                model = module.CAFRN(**kwargs)\n            elif name == "FENet":\n                kwargs = dict(upscale_factor=3)\n                model = module.FENet(**kwargs)\n            elif name == "MHAN":\n                kwargs = dict(in_channels=channels, num_features=64, scale=3)\n                model = module.Net(**kwargs)\n            elif name == "OmniSR":\n                config = yaml.safe_load((root / "train_yamls/train_OmniSR_X3_DIV2K.yaml").read_text())\n                kwargs = dict(num_in_ch=channels, num_out_ch=channels,\n                              num_feat=config["feature_num"], **config["module_params"])\n                model = module.OmniSR(**kwargs)\n            elif name == "SwinIR":\n                kwargs = dict(img_size=64, patch_size=1, in_chans=channels, embed_dim=180,\n                              depths=(6, 6, 6, 6, 6, 6), num_heads=(6, 6, 6, 6, 6, 6),\n                              window_size=8, mlp_ratio=2, upscale=3, img_range=1.0,\n                              upsampler="pixelshuffle", resi_connection="1conv",\n                              use_checkpoint=False)\n                model = module.SwinIR(**kwargs)\n            elif name == "HAT":\n                kwargs = dict(img_size=64, in_chans=channels, embed_dim=144,\n                              depths=(6, 6, 6, 6, 6, 6), num_heads=(6, 6, 6, 6, 6, 6),\n                              window_size=16, compress_ratio=24, squeeze_factor=24,\n                              conv_scale=0.01, overlap_ratio=0.5, mlp_ratio=2,\n                              upscale=3, img_range=1.0, upsampler="pixelshuffle",\n                              resi_connection="1conv", use_checkpoint=False)\n                model = module.HAT(**kwargs)\n            elif name == "RealESRGAN_SRVGG_X3":\n                kwargs = dict(num_in_ch=channels, num_out_ch=channels, num_feat=64,\n                              num_conv=32, upscale=3, act_type="prelu")\n                model = module.SRVGGNetCompact(**kwargs)\n            else:\n                model, kwargs = _classic_model(name, module, channels)\n    finally:\n        if str(import_root) in sys.path:\n            sys.path.remove(str(import_root))\n    spec.update(name=name, constructor=kwargs, input_channels=channels,\n                output_channels=channels, scale=3,\n                parameters=sum(parameter.numel() for parameter in model.parameters()),\n                trainable_parameters=sum(parameter.numel() for parameter in model.parameters()\n                                         if parameter.requires_grad))\n    return model, spec\n\n\ndef load_discriminator(sources: str | Path) -> nn.Module:\n    spec = REGISTRY["RealESRGAN_SRVGG_X3"]\n    root = (Path(sources) / spec["source_dir"]).resolve()\n    verify_source(root, spec)\n    _install_basicsr_stubs()\n    module = _load_file("upstream_realesrgan_discriminator",\n                        root / "realesrgan/archs/discriminator_arch.py")\n    # The adversarial signal is deliberately RGB-only in RGB+NIR experiments.\n    return module.UNetDiscriminatorSN(num_in_ch=3, num_feat=64, skip_connection=True)\n\n\ndef predict(model: nn.Module | None, spec: dict, lr: torch.Tensor) -> torch.Tensor:\n    if model is None:\n        result = torch.nn.functional.interpolate(lr, scale_factor=3, mode="bicubic",\n                                                 align_corners=False)\n    else:\n        multiplier = float(spec["range_multiplier"])\n        offset = float(spec["range_offset"])\n        result = model(lr * multiplier + offset)\n        if not isinstance(result, torch.Tensor):\n            raise TypeError("Unexpected upstream output; auxiliary outputs are not silently selected")\n        result = (result - offset) / multiplier\n    expected = (lr.shape[0], spec["output_channels"], lr.shape[-2] * 3, lr.shape[-1] * 3)\n    if tuple(result.shape) != expected:\n        raise ValueError(f"Model returned {tuple(result.shape)}, expected {expected}")\n    return result\n', 'registry.json': '{\n  "CAFRN": {"repo": "lxy0922/CAFRN", "revision": "fef57e89c46127ddd0ddaaee4599ec39f207e20f", "source_dir": "CAFRN", "entry": "cafrn.py", "provenance": "author implementation", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "extended"},\n  "DCM": {"repo": "Shaosifan/HSENet", "revision": "29e71e33a2d6cb56013886c7f621e793602507f7", "source_dir": "HSENet", "entry": "codes/model/dcm.py", "provenance": "comparison implementation distributed with HSENet; not verified as original DCM author code", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "FENet": {"repo": "wangzheyuan-666/FeNet", "revision": "165954b920c085b07762493193140411e8c4d08a", "source_dir": "FENet", "entry": "model/fenet.py", "provenance": "author Feature Enhancement Network implementation; acronym identity must be checked against the cited paper", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "HAUNet": {"repo": "likakakaka/HAUNet_RSISR", "revision": "53f0b0dec3ecde7e22c328eaf3a73bd8982eaf30", "source_dir": "HAUNet", "entry": "codes/model/haunet.py", "provenance": "author make_model configuration", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "HSENet": {"repo": "Shaosifan/HSENet", "revision": "29e71e33a2d6cb56013886c7f621e793602507f7", "source_dir": "HSENet", "entry": "codes/model/hsenet.py", "provenance": "author implementation", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "MHAN": {"repo": "ZhangDY827/MHAN", "revision": "e5864b078c76817e3bc6eb5af5442ddaeca2b3a7", "source_dir": "MHAN", "entry": "models/model.py", "provenance": "author implementation with published scale=3 interface", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "SAN": {"repo": "daitao/SAN", "revision": "7c1e61f0d17b58e3400a14e631d9185dde1881b6", "source_dir": "SAN", "entry": "TrainCode/model/san.py", "provenance": "author implementation", "range_multiplier": 255, "range_offset": 0, "modalities": ["rgb"], "suite": "extended"},\n  "EDSR": {"repo": "sanghyun-son/EDSR-PyTorch", "revision": "8dba5581a7502b92de9641eb431130d6c8ca5d7f", "source_dir": "EDSR", "entry": "src/model/edsr.py", "provenance": "PyTorch implementation maintained by the EDSR authors; original EDSR configuration", "range_multiplier": 255, "range_offset": 0, "modalities": ["rgb"], "suite": "core"},\n  "RCAN": {"repo": "yulunzhang/RCAN", "revision": "f722e5d0e1eff45148bfaefd9bc85c6f0f26ce02", "source_dir": "RCAN", "entry": "RCAN_TrainCode/code/model/rcan.py", "provenance": "author implementation", "range_multiplier": 255, "range_offset": 0, "modalities": ["rgb"], "suite": "core"},\n  "RDN": {"repo": "sanghyun-son/EDSR-PyTorch", "revision": "8dba5581a7502b92de9641eb431130d6c8ca5d7f", "source_dir": "EDSR", "entry": "src/model/rdn.py", "provenance": "PyTorch port linked by the original RDN authors; RDN-B", "range_multiplier": 255, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "core"},\n  "SwinIR": {"repo": "JingyunLiang/SwinIR", "revision": "6545850fbf8df298df73d81f3e8cba638787c8bd", "source_dir": "SwinIR", "entry": "models/network_swinir.py", "provenance": "author implementation; classical SR large model with native x3 PixelShuffle", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "core"},\n  "HAT": {"repo": "XPixelGroup/HAT", "revision": "1638a9a822581657811867bf670717f8371fc3e5", "source_dir": "HAT", "entry": "hat/archs/hat_arch.py", "provenance": "author HAT-S implementation with native x3 PixelShuffle", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "core"},\n  "OmniSR": {"repo": "Francis0625/Omni-SR", "revision": "cd9901e3e9055b44ec22e6b2c408a08d31ea3dd4", "source_dir": "OmniSR", "entry": "components/OmniSR.py", "provenance": "author X3 DIV2K configuration", "range_multiplier": 2, "range_offset": -1, "modalities": ["rgb", "rgbnir"], "suite": "core"},\n  "RealESRGAN_SRVGG_X3": {"repo": "xinntao/Real-ESRGAN", "revision": "a4abfb2979a7bbff3f69f58f58ae324608821e27", "source_dir": "RealESRGAN", "entry": "realesrgan/archs/srvgg_arch.py", "provenance": "official SRVGGNetCompact and U-Net discriminator, configured for x3 and trained on paired data; an explicit adaptation, not an official Real-ESRGAN x3 checkpoint", "range_multiplier": 1, "range_offset": 0, "modalities": ["rgb", "rgbnir"], "suite": "perceptual", "adaptation": "native arbitrary-scale constructor set to 3; common paired perceptual-GAN training replaces Real-ESRGAN blind degradation"}\n}\n', 'data.py': '"""Locked Mahimandal 160->480 RGB/RGB+NIR dataset reader."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport rasterio\nfrom rasterio.enums import ColorInterp\nimport torch\nfrom torch.utils.data import Dataset\n\n\nSCALE = 3\nLR_SIZE = 160\nHR_SIZE = 480\nBAND_ORDER = ("red", "green", "blue", "nir")\n\n\ndef digest(path: str | Path) -> str:\n    value = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            value.update(chunk)\n    return value.hexdigest()\n\n\ndef resolve(root: str | Path, path: str | Path) -> Path:\n    root = Path(root).resolve()\n    candidate = Path(path)\n    candidate = candidate if candidate.is_absolute() else root / candidate\n    candidate = candidate.resolve()\n    try:\n        candidate.relative_to(root)\n    except ValueError:\n        raise ValueError(f"Path escapes dataset root: {path}")\n    return candidate\n\n\ndef find_release_manifest(root: str | Path) -> Path:\n    root = Path(root)\n    candidates = (\n        root / "paper_reports" / "paper_release_manifest.csv",\n        root / "paper_release_manifest.csv",\n    )\n    found = [path for path in candidates if path.is_file()]\n    if len(found) != 1:\n        raise FileNotFoundError(f"Expected one paper_release_manifest.csv under {root}; found {found}")\n    return found[0]\n\n\ndef read_rows(root: str | Path) -> list[dict]:\n    root = Path(root)\n    with find_release_manifest(root).open(newline="", encoding="utf-8-sig") as stream:\n        rows = list(csv.DictReader(stream))\n    required = {"pair_id", "split", "landscape_class", "spatial_block_id"}\n    missing = required.difference(rows[0] if rows else {})\n    if missing:\n        raise ValueError(f"Release manifest lacks required columns: {sorted(missing)}")\n    for row in rows:\n        pair_id = row["pair_id"]\n        row["lr_path"] = row.get("lr_path") or f"landsat/{pair_id}.tif"\n        row["hr_path"] = row.get("hr_path") or f"sentinel/{pair_id}.tif"\n    return rows\n\n\ndef _band_indices(dataset: rasterio.io.DatasetReader, modality: str) -> list[int]:\n    names = [str(name or "").strip().casefold() for name in dataset.descriptions]\n    aliases = {"red": {"red", "b4", "sr_b4"}, "green": {"green", "b3", "sr_b3"},\n               "blue": {"blue", "b2", "sr_b2"}, "nir": {"nir", "b8", "sr_b5", "b5"}}\n    requested = BAND_ORDER[:3] if modality == "rgb" else BAND_ORDER\n    indices = []\n    for band in requested:\n        matches = [index + 1 for index, value in enumerate(names) if value in aliases[band]]\n        if len(matches) != 1:\n            raise ValueError(f"Cannot resolve one {band} band from descriptions {dataset.descriptions}")\n        indices.append(matches[0])\n    return indices\n\n\ndef _reflectance(array: np.ndarray, source_dtype: str, radiometry: str) -> np.ndarray:\n    array = array.astype(np.float32, copy=False)\n    is_integer = np.issubdtype(np.dtype(source_dtype), np.integer)\n    if radiometry == "normalized_float32" and is_integer:\n        raise ValueError("normalized_float32 requires floating-point TIFF storage")\n    if is_integer:\n        array = array / 10000.0\n    elif np.nanmax(array) > 1.5:\n        raise ValueError("Floating TIFF exceeds reflectance range; scaling is ambiguous")\n    array = np.clip(array, 0.0, 1.0)\n    if radiometry == "quantized8":\n        array = np.rint(array * 255.0) / 255.0\n    elif radiometry not in {"reflectance16", "normalized_float32"}:\n        raise ValueError(f"Unknown radiometry protocol: {radiometry}")\n    return array\n\n\nclass Pairs(Dataset):\n    def __init__(self, root: str | Path, split: str, modality: str = "rgb",\n                 radiometry: str = "reflectance16", augment: bool = False,\n                 lr_crop: int | None = None):\n        if modality not in {"rgb", "rgbnir"}:\n            raise ValueError("modality must be rgb or rgbnir")\n        if lr_crop is not None and not 1 <= lr_crop <= LR_SIZE:\n            raise ValueError(f"lr_crop must be in [1,{LR_SIZE}]")\n        self.root = Path(root)\n        self.modality = modality\n        self.radiometry = radiometry\n        self.augment = augment\n        self.lr_crop = lr_crop\n        self.rows = sorted((row for row in read_rows(root) if row["split"] == split),\n                           key=lambda row: row["pair_id"])\n        if not self.rows:\n            raise ValueError(f"Empty split {split!r}")\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int):\n        row = self.rows[index]\n        with rasterio.open(resolve(self.root, row["lr_path"])) as source:\n            lr = _reflectance(source.read(_band_indices(source, self.modality)), source.dtypes[0], self.radiometry)\n            lr_mask = np.all(source.read_masks() > 0, axis=0)\n        with rasterio.open(resolve(self.root, row["hr_path"])) as source:\n            hr = _reflectance(source.read(_band_indices(source, self.modality)), source.dtypes[0], self.radiometry)\n            hr_mask = np.all(source.read_masks() > 0, axis=0)\n        channels = 3 if self.modality == "rgb" else 4\n        if lr.shape != (channels, LR_SIZE, LR_SIZE) or hr.shape != (channels, HR_SIZE, HR_SIZE):\n            raise ValueError(f"Unexpected geometry for {row[\'pair_id\']}: {lr.shape}, {hr.shape}")\n        joint_mask = hr_mask & np.repeat(np.repeat(lr_mask, SCALE, axis=0), SCALE, axis=1)\n        if not joint_mask.any() or not np.isfinite(lr).all() or not np.isfinite(hr).all():\n            raise ValueError(f"No finite jointly valid pixels for {row[\'pair_id\']}")\n        if self.lr_crop and self.lr_crop < LR_SIZE:\n            top = int(torch.randint(0, LR_SIZE - self.lr_crop + 1, ()).item())\n            left = int(torch.randint(0, LR_SIZE - self.lr_crop + 1, ()).item())\n            bottom, right = top + self.lr_crop, left + self.lr_crop\n            lr = lr[:, top:bottom, left:right]\n            hr = hr[:, top * SCALE:bottom * SCALE, left * SCALE:right * SCALE]\n            joint_mask = joint_mask[top * SCALE:bottom * SCALE, left * SCALE:right * SCALE]\n        arrays = [lr, hr, joint_mask[None].astype(np.float32)]\n        if self.augment:\n            rotations = int(torch.randint(4, ()).item())\n            arrays = [np.rot90(array, rotations, axes=(-2, -1)) for array in arrays]\n            if torch.rand(()) < 0.5:\n                arrays = [array[..., ::-1] for array in arrays]\n        return (*[torch.from_numpy(array.copy()) for array in arrays], index)\n\n\ndef audit(root: str | Path) -> dict:\n    root = Path(root)\n    rows = read_rows(root)\n    counts = Counter(row["split"] for row in rows)\n    if set(counts) != {"train", "val", "test"} or min(counts.values()) < 1:\n        raise ValueError(f"Release requires nonempty train/val/test: {dict(counts)}")\n    if len({row["pair_id"] for row in rows}) != len(rows):\n        raise ValueError("Release manifest contains duplicate pair IDs")\n    blocks: dict[str, set[str]] = {}\n    classes: dict[str, Counter] = {split: Counter() for split in counts}\n    regions: dict[str, Counter] = {split: Counter() for split in counts}\n    source_dtypes: set[str] = set()\n    for row in rows:\n        blocks.setdefault(row["spatial_block_id"], set()).add(row["split"])\n        classes[row["split"]][row.get("landscape_class", "unknown")] += 1\n        regions[row["split"]][row.get("geographic_stratum", "unknown")] += 1\n        for field, shape, resolution in (("lr_path", (LR_SIZE, LR_SIZE), 30.0),\n                                         ("hr_path", (HR_SIZE, HR_SIZE), 10.0)):\n            path = resolve(root, row[field])\n            if not path.is_file():\n                raise FileNotFoundError(path)\n            with rasterio.open(path) as source:\n                if (source.height, source.width) != shape or source.count != 4:\n                    raise ValueError(f"Unexpected TIFF contract: {path}")\n                if not np.allclose(source.res, (resolution, resolution), atol=1e-6):\n                    raise ValueError(f"Unexpected resolution: {path}: {source.res}")\n                source_dtypes.update(source.dtypes)\n                _band_indices(source, "rgbnir")\n    leaked = {block: sorted(splits) for block, splits in blocks.items() if len(splits) > 1}\n    if leaked:\n        raise ValueError(f"Spatial blocks cross release splits: {list(leaked.items())[:5]}")\n    protocol_path = root / "protocol.json"\n    protocol = json.loads(protocol_path.read_text(encoding="utf-8")) if protocol_path.is_file() else {}\n    return {\n        "pairs": len(rows), "counts": dict(counts),\n        "realized_fractions": {key: value / len(rows) for key, value in counts.items()},\n        "intended_split_fractions": protocol.get(\n            "release_split_objective",\n            protocol.get("split_fractions", {"train": 0.78, "val": 0.11, "test": 0.11}),\n        ),\n        "split_note": (\n            "The release allocator objective is 78/11/11; complete 100-km "\n            "blocks produce the locked realized counts."\n        ),\n        "geometry": {"lr": [4, LR_SIZE, LR_SIZE], "hr": [4, HR_SIZE, HR_SIZE], "scale": SCALE},\n        "source_band_order": ["blue", "green", "red", "nir"],\n        "model_band_order": list(BAND_ORDER),\n        "source_dtypes": sorted(source_dtypes),\n        "reflectance_decode": ("clip(source/10000,0,1)" if source_dtypes == {"int16"}\n                               else "floating reflectance clipped to [0,1]"),\n        "classes_by_split": {key: dict(value) for key, value in classes.items()},\n        "geographic_strata_by_split": {key: dict(value) for key, value in regions.items()},\n        "release_manifest": str(find_release_manifest(root)),\n        "release_manifest_sha256": digest(find_release_manifest(root)),\n        "protocol_sha256": digest(protocol_path) if protocol_path.is_file() else None,\n        "spatial_blocks": len(blocks), "cross_split_spatial_blocks": 0,\n    }\n\n\ndef save_prediction(path: str | Path, image: np.ndarray, mask: np.ndarray,\n                    reference: str | Path, modality: str, tags: dict) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    names = BAND_ORDER[:3] if modality == "rgb" else BAND_ORDER\n    with rasterio.open(reference) as source:\n        profile = source.profile.copy()\n        metadata = source.tags()\n    profile.update(count=len(names), dtype="float32", compress="deflate", predictor=3, nodata=None)\n    temporary = path.with_suffix(".tmp.tif")\n    with rasterio.Env(GDAL_TIFF_INTERNAL_MASK=True):\n        with rasterio.open(temporary, "w", **profile) as output:\n            output.write(image.astype(np.float32))\n            output.write_mask(mask.squeeze().astype(np.uint8) * 255)\n            metadata.update(sensor="model_prediction", band_order=",".join(names), **tags)\n            output.update_tags(**{key: str(value) for key, value in metadata.items()})\n            if modality == "rgb":\n                output.colorinterp = (ColorInterp.red, ColorInterp.green, ColorInterp.blue)\n            for index, name in enumerate(names, 1):\n                output.set_band_description(index, name)\n    temporary.replace(path)\n', 'metrics.py': '"""Common full-reference metrics for reflectance-domain cross-sensor SR."""\nfrom __future__ import annotations\n\nimport math\n\nimport torch\nfrom torch.nn import functional as F\n\n\ndef _weighted_mean(value: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    if mask.shape[1] == 1 and value.shape[1] != 1:\n        mask = mask.expand(-1, value.shape[1], -1, -1)\n    return (value * mask).sum() / mask.sum().clamp_min(1)\n\n\ndef mae(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    return _weighted_mean((prediction - target).abs(), mask)\n\n\ndef mse(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    return _weighted_mean((prediction - target).square(), mask)\n\n\ndef psnr(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    return -10.0 * torch.log10(mse(prediction, target, mask).clamp_min(1e-12))\n\n\ndef _gaussian_kernel(channels: int, device, dtype) -> torch.Tensor:\n    axis = torch.arange(11, device=device, dtype=dtype) - 5\n    gaussian = torch.exp(-(axis.square()) / (2 * 1.5**2))\n    gaussian = gaussian / gaussian.sum()\n    kernel = torch.outer(gaussian, gaussian)[None, None]\n    return kernel.expand(channels, 1, 11, 11).contiguous()\n\n\ndef ssim(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    channels = prediction.shape[1]\n    kernel = _gaussian_kernel(channels, prediction.device, prediction.dtype)\n    mu_x = F.conv2d(prediction, kernel, padding=5, groups=channels)\n    mu_y = F.conv2d(target, kernel, padding=5, groups=channels)\n    sigma_x = F.conv2d(prediction.square(), kernel, padding=5, groups=channels) - mu_x.square()\n    sigma_y = F.conv2d(target.square(), kernel, padding=5, groups=channels) - mu_y.square()\n    sigma_xy = F.conv2d(prediction * target, kernel, padding=5, groups=channels) - mu_x * mu_y\n    c1, c2 = 0.01**2, 0.03**2\n    value = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / (\n        (mu_x.square() + mu_y.square() + c1) * (sigma_x + sigma_y + c2)\n    ).clamp_min(1e-12)\n    return _weighted_mean(value, mask)\n\n\ndef _correlation(first: torch.Tensor, second: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:\n    if mask.shape[1] == 1 and first.shape[1] != 1:\n        mask = mask.expand(-1, first.shape[1], -1, -1)\n    weight = mask.sum().clamp_min(1)\n    first_mean = (first * mask).sum() / weight\n    second_mean = (second * mask).sum() / weight\n    first_centered = (first - first_mean) * mask\n    second_centered = (second - second_mean) * mask\n    return (first_centered * second_centered).sum() / torch.sqrt(\n        first_centered.square().sum() * second_centered.square().sum()\n    ).clamp_min(1e-12)\n\n\ndef _sobel(image: torch.Tensor) -> torch.Tensor:\n    gray = image[:, :3].mean(dim=1, keepdim=True)\n    x_kernel = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],\n                            device=image.device, dtype=image.dtype)[None, None] / 8\n    y_kernel = x_kernel.transpose(-1, -2)\n    dx = F.conv2d(gray, x_kernel, padding=1)\n    dy = F.conv2d(gray, y_kernel, padding=1)\n    return torch.sqrt(dx.square() + dy.square() + 1e-12)\n\n\ndef edge_f1(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor,\n            threshold: float = 0.04) -> torch.Tensor:\n    predicted = (_sobel(prediction) >= threshold).float() * mask\n    reference = (_sobel(target) >= threshold).float() * mask\n    predicted_dilated = F.max_pool2d(predicted, 3, stride=1, padding=1)\n    reference_dilated = F.max_pool2d(reference, 3, stride=1, padding=1)\n    precision = (predicted * reference_dilated).sum() / predicted.sum().clamp_min(1)\n    recall = (reference * predicted_dilated).sum() / reference.sum().clamp_min(1)\n    return 2 * precision * recall / (precision + recall).clamp_min(1e-12)\n\n\ndef _laplacian(image: torch.Tensor) -> torch.Tensor:\n    channels = image.shape[1]\n    kernel = torch.tensor([[0, 1, 0], [1, -4, 1], [0, 1, 0]],\n                          device=image.device, dtype=image.dtype)[None, None]\n    return F.conv2d(image, kernel.expand(channels, 1, 3, 3), padding=1, groups=channels)\n\n\ndef _crop(values: tuple[torch.Tensor, ...], border: int) -> tuple[torch.Tensor, ...]:\n    if border == 0:\n        return values\n    return tuple(value[..., border:-border, border:-border] for value in values)\n\n\ndef _y(image: torch.Tensor) -> torch.Tensor:\n    weights = image.new_tensor((0.299, 0.587, 0.114)).view(1, 3, 1, 1)\n    return (image[:, :3] * weights).sum(dim=1, keepdim=True)\n\n\ndef _ndvi(image: torch.Tensor) -> torch.Tensor:\n    red, nir = image[:, 0:1], image[:, 3:4]\n    return (nir - red) / (nir + red).clamp_min(1e-6)\n\n\ndef metric_bundle(prediction: torch.Tensor, target: torch.Tensor, mask: torch.Tensor,\n                  border: int = 3, lpips_model=None) -> dict[str, float]:\n    prediction, target, mask = prediction.float(), target.float(), mask.float()\n    prediction, target, mask = _crop((prediction, target, mask), border)\n    channels = prediction.shape[1]\n    values = {\n        "mae": float(mae(prediction, target, mask)),\n        "rmse": float(torch.sqrt(mse(prediction, target, mask))),\n        "psnr": float(psnr(prediction, target, mask)),\n        "ssim": float(ssim(prediction, target, mask)),\n        "edge_f1": float(edge_f1(prediction, target, mask)),\n        "edge_psnr": float(psnr(_sobel(prediction), _sobel(target), mask)),\n        "gradient_correlation": float(_correlation(_sobel(prediction), _sobel(target), mask)),\n        "scc": float(_correlation(_laplacian(prediction), _laplacian(target), mask)),\n    }\n    rgb_prediction, rgb_target = prediction[:, :3], target[:, :3]\n    values.update(\n        rgb_psnr=float(psnr(rgb_prediction, rgb_target, mask)),\n        rgb_ssim=float(ssim(rgb_prediction, rgb_target, mask)),\n        y_psnr=float(psnr(_y(rgb_prediction), _y(rgb_target), mask)),\n        y_ssim=float(ssim(_y(rgb_prediction), _y(rgb_target), mask)),\n    )\n    names = ("red", "green", "blue", "nir")[:channels]\n    band_mse, band_uiqi, band_means = [], [], []\n    for index, name in enumerate(names):\n        predicted_band, target_band = prediction[:, index:index + 1], target[:, index:index + 1]\n        values[f"{name}_psnr"] = float(psnr(predicted_band, target_band, mask))\n        error = mse(predicted_band, target_band, mask)\n        band_mse.append(error)\n        band_means.append(_weighted_mean(target_band, mask).abs().clamp_min(1e-6))\n        predicted_mean = _weighted_mean(predicted_band, mask)\n        target_mean = _weighted_mean(target_band, mask)\n        predicted_variance = _weighted_mean((predicted_band - predicted_mean).square(), mask)\n        target_variance = _weighted_mean((target_band - target_mean).square(), mask)\n        covariance = _weighted_mean((predicted_band - predicted_mean) * (target_band - target_mean), mask)\n        band_uiqi.append((4 * covariance * predicted_mean * target_mean) /\n                         ((predicted_variance + target_variance) *\n                          (predicted_mean.square() + target_mean.square())).clamp_min(1e-12))\n    values["ergas"] = float((100 / 3) * torch.sqrt(torch.stack([\n        error / mean.square() for error, mean in zip(band_mse, band_means)\n    ]).mean()))\n    values["uiqi"] = float(torch.stack(band_uiqi).mean())\n    dot = (prediction * target).sum(dim=1)\n    denominator = torch.linalg.vector_norm(prediction, dim=1) * torch.linalg.vector_norm(target, dim=1)\n    angle = torch.acos((dot / denominator.clamp_min(1e-8)).clamp(-1 + 1e-7, 1 - 1e-7))\n    values["sam_degrees"] = float(_weighted_mean(angle[:, None], mask) * 180 / math.pi)\n    if channels == 4:\n        predicted_ndvi, target_ndvi = _ndvi(prediction), _ndvi(target)\n        values.update(ndvi_mae=float(mae(predicted_ndvi, target_ndvi, mask)),\n                      ndvi_rmse=float(torch.sqrt(mse(predicted_ndvi, target_ndvi, mask))),\n                      ndvi_correlation=float(_correlation(predicted_ndvi, target_ndvi, mask)))\n    if lpips_model is not None:\n        # Invalid pixels are replaced by the target before the RGB feature comparison.\n        lpips_mask = mask.expand(-1, 3, -1, -1)\n        safe_prediction = rgb_prediction * lpips_mask + rgb_target * (1 - lpips_mask)\n        values["lpips"] = float(lpips_model(safe_prediction * 2 - 1, rgb_target * 2 - 1).mean())\n    return values\n', 'cross_dataset_models.py': '"""Full-frame RGB transfer study for pinned published benchmark models.\n\nThe source checkpoints stay immutable. OLI2MSI validation is derived only from\nits official training directory; the official 100-pair test directory is locked.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport rasterio\nimport torch\nfrom torch.nn import functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nfrom adapters import load as load_benchmark, load_discriminator, predict as predict_benchmark\nfrom data import Pairs as MahimandalPairs, digest\nfrom metrics import mae, metric_bundle, psnr\nPROFILES = {"rgb": (3, 3)}\n\n\ndef write_json(path: Path, value: dict) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef write_csv(path: Path, rows: list[dict]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(dict.fromkeys(k for r in rows for k in r)))\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef oli_layout(root: Path) -> dict[str, Path]:\n    aliases = {"trainlr": "train_lr", "trainhr": "train_hr",\n               "testlr": "test_lr", "testhr": "test_hr"}\n    found = {}\n    for child in root.iterdir():\n        if child.is_dir():\n            name = "".join(c for c in child.name.casefold() if c.isalnum())\n            if name in aliases:\n                found[aliases[name]] = child\n    if set(found) != set(aliases.values()):\n        raise ValueError(f"Expected train_lr/train_hr/test_lr/test_hr under {root}")\n    return found\n\n\ndef oli_inventory(root: Path, seed: int = 42, val_percent: int = 10) -> dict:\n    layout = oli_layout(root)\n    inventory = {}\n    manifest = hashlib.sha256()\n    for split in ("train", "test"):\n        sides = {}\n        for side in ("lr", "hr"):\n            paths = [p for p in layout[f"{split}_{side}"].iterdir()\n                     if p.is_file() and p.suffix.casefold() in {".tif", ".tiff"}]\n            files = {p.stem.casefold(): p for p in paths}\n            if len(files) != len(paths):\n                raise ValueError(f"Duplicate pair identifiers in {layout[split + \'_\' + side]}")\n            sides[side] = files\n        if not sides["lr"] or sides["lr"].keys() != sides["hr"].keys():\n            raise ValueError(f"Unpaired or empty OLI2MSI {split} files")\n        inventory[split] = []\n        for name in sorted(sides["lr"]):\n            lr, hr = sides["lr"][name], sides["hr"][name]\n            manifest.update(f"{split}|{name}|{lr.stat().st_size}|{hr.stat().st_size}\\n".encode())\n            inventory[split].append((name, lr, hr))\n    if {row[0] for row in inventory["train"]} & {row[0] for row in inventory["test"]}:\n        raise ValueError("OLI2MSI train and test share pair identifiers")\n    with rasterio.open(inventory["train"][0][1]) as lr, rasterio.open(inventory["train"][0][2]) as hr:\n        channels = min(lr.count, hr.count)\n        descriptions = (lr.descriptions, hr.descriptions)\n        dtypes = (lr.dtypes[0], hr.dtypes[0])\n    nir_available = channels >= 4 and all(\n        name and "nir" in name.casefold() for name in (descriptions[0][3], descriptions[1][3])\n    )\n    return {"rows": inventory, "identity": manifest.hexdigest(), "channels": channels,\n            "nir_available": nir_available, "descriptions": descriptions, "dtypes": dtypes,\n            "counts": {"official_train": len(inventory["train"]),\n                       "official_test": len(inventory["test"])},\n            "seed": seed, "validation_percent": val_percent}\n\n\ndef oli_split(inventory: dict, split: str) -> list[tuple[str, Path, Path]]:\n    if split == "test":\n        return inventory["rows"]["test"]\n    if split not in {"train", "val"}:\n        raise ValueError(split)\n    result = []\n    for item in inventory["rows"]["train"]:\n        value = hashlib.sha256(f"{inventory[\'seed\']}:{item[0]}".encode()).digest()\n        assigned = "val" if int.from_bytes(value[:4], "big") % 100 < inventory["validation_percent"] else "train"\n        if assigned == split:\n            result.append(item)\n    return result\n\n\ndef decode(image: np.ndarray, dtype: str, radiometry: str) -> np.ndarray:\n    if np.dtype(dtype) == np.dtype("uint8"):\n        image = image.astype(np.float32) / 255.0\n    elif np.issubdtype(np.dtype(dtype), np.integer):\n        image = image.astype(np.float32) / 10000.0\n    else:\n        image = image.astype(np.float32)\n        if np.nanmax(image) > 1.5:\n            raise ValueError("Floating OLI2MSI reflectance exceeds 1.5; inspect source scaling")\n    image = np.clip(image, 0, 1)\n    if radiometry == "quantized8":\n        image = np.rint(image * 255) / 255\n    elif radiometry != "reflectance16":\n        raise ValueError(radiometry)\n    return image\n\n\nclass OliPairs(Dataset):\n    def __init__(self, inventory: dict, split: str, modality: str,\n                 radiometry: str, augment: bool = False):\n        self.rows = oli_split(inventory, split)\n        self.modality, self.radiometry, self.augment = modality, radiometry, augment\n        if not self.rows:\n            raise ValueError(f"Empty OLI2MSI {split} split")\n        if modality != "rgb" and not inventory["nir_available"]:\n            raise ValueError("OLI2MSI lacks verified paired NIR; RGB+NIR experiments cannot run")\n\n    def __len__(self) -> int:\n        return len(self.rows)\n\n    def __getitem__(self, index: int):\n        name, lr_path, hr_path = self.rows[index]\n        bands = [1, 2, 3] if self.modality == "rgb" else [1, 2, 3, 4]\n        arrays = []\n        for path, size in ((lr_path, 160), (hr_path, 480)):\n            with rasterio.open(path) as src:\n                if src.width != size or src.height != size:\n                    raise ValueError(f"Expected {size}x{size}: {path}")\n                if src.count < len(bands):\n                    raise ValueError(f"Missing bands in {path}")\n                raw = src.read(bands)\n                image = decode(raw, src.dtypes[0], self.radiometry)\n                mask = np.all(src.read_masks(bands) > 0, axis=0) & np.isfinite(raw).all(axis=0)\n                arrays.append((image, mask))\n        (lr, lr_mask), (hr, hr_mask) = arrays\n        mask = hr_mask & np.repeat(np.repeat(lr_mask, 3, 0), 3, 1)\n        if not mask.any():\n            raise ValueError(f"No jointly valid pixels: {name}")\n        payload = [lr, hr, mask[None].astype(np.float32)]\n        if self.augment:\n            rotation = int(torch.randint(4, ()).item())\n            payload = [np.rot90(x, rotation, axes=(-2, -1)) for x in payload]\n            if torch.rand(()) < 0.5:\n                payload = [x[..., ::-1] for x in payload]\n        return (*[torch.from_numpy(x.copy()) for x in payload], name)\n\n\nclass StudyPairs(Dataset):\n    def __init__(self, domain: str, root: Path, split: str, modality: str,\n                 radiometry: str, inventory: dict | None = None, augment: bool = False):\n        self.modality = modality\n        channels = "rgb" if modality == "rgb" else "rgbnir"\n        if domain == "oli2msi":\n            self.source = OliPairs(inventory or oli_inventory(root), split, channels, radiometry, augment)\n        elif domain == "mahimandal":\n            self.source = MahimandalPairs(root, split, channels, radiometry, augment, 160)\n        else:\n            raise ValueError(domain)\n\n    def __len__(self) -> int:\n        return len(self.source)\n\n    def __getitem__(self, index: int):\n        lr, hr, mask, row = self.source[index]\n        if self.modality == "rgbnir_rgb":\n            hr = hr[:3]\n        pair_id = row if isinstance(row, str) else self.source.rows[index]["pair_id"]\n        return lr, hr, mask, pair_id\n\n\ndef load_model(kind: str, name: str, sources: Path, modality: str,\n               architecture: Path | None, width: int):\n    if kind == "benchmark":\n        model, spec = load_benchmark(name, sources, modality)\n        return model, spec\n    raise ValueError(kind)\n\n\ndef predict(kind: str, model, spec: dict, lr: torch.Tensor):\n    if model is None:\n        return F.interpolate(lr, scale_factor=3, mode="bicubic", align_corners=False)\n    if kind == "benchmark":\n        return predict_benchmark(model, spec, lr)\n    return model(lr).image\n\n\ndef benchmark_checkpoint_name(saved_protocol: dict) -> str | None:\n    """Read both OLI-trained and original benchmark checkpoint identities."""\n    names = {saved_protocol.get("model_name")}\n    for key in ("model_spec", "model"):\n        value = saved_protocol.get(key)\n        if isinstance(value, dict):\n            names.add(value.get("name"))\n    names.discard(None)\n    if len(names) > 1:\n        raise ValueError(f"Conflicting checkpoint benchmark identities: {names}")\n    return next(iter(names), None)\n\n\ndef protocol(args, identity: str, spec: dict) -> dict:\n    value = {"source_dataset": "oli2msi", "dataset_identity": identity,\n            "model_kind": args.kind, "model_name": args.model, "model_spec": spec,\n            "modality": args.modality, "radiometry": args.radiometry,\n            "seed": args.seed, "batch_size": args.batch_size, "lr": args.lr,\n            "optimizer": "Adam", "loss": (\n                "masked L1 + RGB LPIPS + logistic GAN after warmup"\n                if args.kind == "benchmark" and args.model == "RealESRGAN_SRVGG_X3"\n                else "masked L1"), "augmentation": "D4",\n            "train_geometry": [160, 480], "selection": "full-frame OLI2MSI validation PSNR"}\n    if args.kind == "benchmark" and args.model == "RealESRGAN_SRVGG_X3":\n        value["gan_warmup_epochs"] = max(1, args.epochs // 3)\n    return json.loads(json.dumps(value, allow_nan=False))\n\n\ndef training_root(args) -> Path:\n    return args.output / "trained_oli2msi" / args.kind / args.radiometry / args.modality / args.model / f"seed_{args.seed}"\n\n\ndef train(args, inventory: dict) -> Path:\n    root = training_root(args)\n    root.mkdir(parents=True, exist_ok=True)\n    torch.manual_seed(args.seed)\n    np.random.seed(args.seed)\n    random.seed(args.seed)\n    model, spec = load_model(args.kind, args.model, args.sources, args.modality,\n                             args.architecture, args.width)\n    if model is None:\n        raise ValueError("Bicubic is an evaluation baseline")\n    signature = protocol(args, inventory["identity"], spec)\n    lock = root / "protocol.json"\n    if lock.is_file() and json.loads(lock.read_text()) != signature:\n        raise ValueError(f"Existing training protocol differs: {lock}")\n    write_json(lock, signature)\n    device = torch.device(args.device)\n    model.to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)\n    use_gan = args.kind == "benchmark" and args.model == "RealESRGAN_SRVGG_X3"\n    discriminator = discriminator_optimizer = perceptual = None\n    if use_gan:\n        import lpips\n        discriminator = load_discriminator(args.sources).to(device)\n        discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=args.lr)\n        perceptual = lpips.LPIPS(net="alex").to(device).eval()\n        for parameter in perceptual.parameters():\n            parameter.requires_grad_(False)\n    training = StudyPairs("oli2msi", args.oli, "train", args.modality, args.radiometry, inventory, True)\n    validation = StudyPairs("oli2msi", args.oli, "val", args.modality, args.radiometry, inventory)\n    last, best = root / "last.pt", root / "best.pt"\n    epoch0, best_psnr, stale = 0, -float("inf"), 0\n    if last.is_file():\n        state = torch.load(last, map_location="cpu", weights_only=False)\n        if state["protocol"] != signature:\n            raise ValueError("Checkpoint protocol differs from current request")\n        model.load_state_dict(state["model"], strict=True)\n        optimizer.load_state_dict(state["optimizer"])\n        if use_gan:\n            discriminator.load_state_dict(state["discriminator"], strict=True)\n            discriminator_optimizer.load_state_dict(state["discriminator_optimizer"])\n        epoch0, best_psnr, stale = state["epoch"], state["best_psnr"], state["stale"]\n    for epoch in range(epoch0 + 1, args.epochs + 1):\n        if stale >= args.patience:\n            break\n        torch.manual_seed(args.seed + epoch)\n        loader = DataLoader(training, batch_size=args.batch_size, shuffle=True,\n                            num_workers=args.workers, pin_memory=device.type == "cuda")\n        model.train()\n        losses = []\n        for lr, hr, mask, _ in loader:\n            lr, hr, mask = lr.to(device), hr.to(device), mask.to(device)\n            output = predict(args.kind, model, spec, lr)\n            loss = mae(output, hr, mask)\n            if use_gan and epoch > max(1, args.epochs // 3):\n                discriminator_optimizer.zero_grad(set_to_none=True)\n                real_logits = discriminator(hr[:, :3])\n                fake_logits = discriminator(output[:, :3].detach())\n                d_loss = F.softplus(-real_logits).mean() + F.softplus(fake_logits).mean()\n                d_loss.backward()\n                discriminator_optimizer.step()\n                for parameter in discriminator.parameters():\n                    parameter.requires_grad_(False)\n                loss = (loss + 0.1 * perceptual(output[:, :3] * 2 - 1,\n                                                 hr[:, :3] * 2 - 1).mean()\n                        + 0.005 * F.softplus(-discriminator(output[:, :3])).mean())\n                for parameter in discriminator.parameters():\n                    parameter.requires_grad_(True)\n            optimizer.zero_grad(set_to_none=True)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1)\n            optimizer.step()\n            losses.append(float(loss.detach()))\n        model.eval()\n        scores = []\n        with torch.no_grad():\n            for lr, hr, mask, _ in DataLoader(validation, batch_size=1):\n                output = predict(args.kind, model, spec, lr.to(device)).float().clamp(0, 1).cpu()\n                scores.append(float(psnr(output, hr, mask)))\n        score = float(np.mean(scores))\n        improved = score > best_psnr\n        stale = 0 if improved else stale + 1\n        best_psnr = max(best_psnr, score)\n        state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),\n                 "epoch": epoch, "best_psnr": best_psnr, "stale": stale,\n                 "protocol": signature}\n        if use_gan:\n            state["discriminator"] = discriminator.state_dict()\n            state["discriminator_optimizer"] = discriminator_optimizer.state_dict()\n        temporary = root / "checkpoint.tmp"\n        torch.save(state, temporary)\n        temporary.replace(last)\n        if improved:\n            torch.save(state, temporary)\n            temporary.replace(best)\n        history = root / "training.csv"\n        row = {"epoch": epoch, "train_l1": float(np.mean(losses)), "val_psnr": score,\n               "best_val_psnr": best_psnr, "stale": stale,\n               "phase": "perceptual_gan" if use_gan and epoch > max(1, args.epochs // 3) else "pixel"}\n        new_file = not history.is_file()\n        with history.open("a", newline="", encoding="utf-8") as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(row))\n            if new_file:\n                writer.writeheader()\n            writer.writerow(row)\n        print(f"{args.kind}/{args.model}/{args.modality}: epoch {epoch}/{args.epochs} "\n              f"train_l1={np.mean(losses):.6f} val_psnr={score:.4f} stale={stale}/{args.patience}", flush=True)\n    if not best.is_file():\n        raise FileNotFoundError(f"No best checkpoint: {best}")\n    write_json(root / "training_status.json", {\n        "best_validation_psnr": best_psnr, "early_stopped": stale >= args.patience,\n        "best_checkpoint_sha256": digest(best), "patience": args.patience,\n        "epoch_ceiling": args.epochs,\n        "history_note": "Only epochs executed by this runner are logged; older missing history is not reconstructed."})\n    return best\n\n\n@torch.no_grad()\ndef evaluate(args, inventory: dict, checkpoint: Path, domain: str) -> dict:\n    model, spec = load_model(args.kind, args.model, args.sources, args.modality,\n                             args.architecture, args.width)\n    if model is not None:\n        state = torch.load(checkpoint, map_location="cpu", weights_only=False)\n        if state["protocol"].get("modality") != args.modality:\n            raise ValueError("Checkpoint modality does not match request")\n        if state["protocol"].get("radiometry") != args.radiometry:\n            raise ValueError("Checkpoint radiometry does not match request")\n        if args.kind == "nas":\n            expected_architecture = digest(args.architecture)\n            saved_architecture = state["protocol"].get(\n                "architecture_sha256", state["protocol"].get("model_spec", {}).get("architecture_sha256")\n            )\n            if saved_architecture != expected_architecture:\n                raise ValueError("Checkpoint architecture does not match request")\n        if args.kind == "benchmark" and benchmark_checkpoint_name(state["protocol"]) != args.model:\n            raise ValueError("Checkpoint benchmark architecture does not match request")\n        model.load_state_dict(state["model"], strict=True)\n    checkpoint_hash = digest(checkpoint) if model is not None else "bicubic"\n    target_root = args.oli if domain == "oli2msi" else args.mahimandal\n    target_identity = inventory["identity"] if domain == "oli2msi" else digest(target_root / "paper_release_manifest.csv")\n    destination = args.output / "evaluation" / args.trained_on / domain / args.kind / args.radiometry / args.modality / args.model / checkpoint_hash[:12]\n    receipt = {"trained_on": args.trained_on, "tested_on": domain,\n               "source_checkpoint_sha256": checkpoint_hash, "target_identity": target_identity,\n               "modality": args.modality, "radiometry": args.radiometry,\n               "geometry": [160, 480], "metric_domain": "reflectance [0,1]",\n               "border_crop": args.border_crop}\n    lock = destination / "protocol.json"\n    if lock.is_file() and json.loads(lock.read_text()) != receipt:\n        raise ValueError(f"Existing evaluation differs: {lock}")\n    write_json(lock, receipt)\n    result = destination / "summary.json"\n    if result.is_file() and (destination / "per_image.csv").is_file():\n        print("Reused:", result, flush=True)\n        return json.loads(result.read_text())\n    device = torch.device(args.device)\n    if model is not None:\n        model.to(device).eval()\n    dataset = StudyPairs(domain, target_root, "test", args.modality,\n                         args.radiometry, inventory if domain == "oli2msi" else None)\n    rows = []\n    pictured_classes = set()\n    for index, (lr, hr, mask, pair_id) in enumerate(DataLoader(dataset, batch_size=1)):\n        if device.type == "cuda":\n            torch.cuda.synchronize()\n        started = time.perf_counter()\n        output = predict(args.kind, model, spec, lr.to(device)).float().clamp(0, 1)\n        if device.type == "cuda":\n            torch.cuda.synchronize()\n        elapsed = (time.perf_counter() - started) * 1000\n        if tuple(output.shape) != tuple(hr.shape) or not torch.isfinite(output).all():\n            raise ValueError(f"Invalid prediction for {pair_id[0]}")\n        measures = metric_bundle(output, hr.to(device), mask.to(device), border=args.border_crop)\n        metadata = dataset.source.rows[index] if domain == "mahimandal" else {}\n        scene = metadata.get("landscape_class", "unlabeled")\n        rows.append({"pair_id": str(pair_id[0]),\n                     "landscape_class": scene,\n                     "geographic_stratum": metadata.get("geographic_stratum", "unlabeled"),\n                     "spatial_block_id": metadata.get("spatial_block_id", ""),\n                     "day_gap": metadata.get("day_gap", ""),\n                     "inference_ms": elapsed, **measures})\n        first_of_class = domain == "mahimandal" and scene not in pictured_classes\n        if index < 5 or first_of_class:\n            import matplotlib.pyplot as plt\n            name = (f"class_{\'\'.join(c if c.isalnum() else \'_\' for c in scene)}.png"\n                    if first_of_class else f"test_{index:04d}.png")\n            visual = destination / "figures" / name\n            visual.parent.mkdir(parents=True, exist_ok=True)\n            lr_rgb = F.interpolate(lr[:, :3], scale_factor=3, mode="nearest")[0].numpy()\n            target_rgb = hr[0, :3].numpy()\n            prediction_rgb = output[0, :3].cpu().numpy()\n            panels = (lr_rgb, target_rgb, prediction_rgb,\n                      np.abs(prediction_rgb - target_rgb))\n            figure, axes = plt.subplots(1, 4, figsize=(16, 4))\n            for axis, image, title in zip(axes, panels,\n                                          ("Landsat LR", "Sentinel HR", args.model, "RGB absolute error")):\n                shown = image.transpose(1, 2, 0)\n                axis.imshow(np.clip(shown / 0.3, 0, 1) if title != "RGB absolute error"\n                            else np.clip(shown / 0.1, 0, 1))\n                axis.set_title(title)\n                axis.axis("off")\n            figure.suptitle(f"{domain} test/{index}: {pair_id[0]} | {scene} | display stretch only")\n            figure.tight_layout()\n            figure.savefig(visual, dpi=140)\n            plt.close(figure)\n            pictured_classes.add(scene)\n    write_csv(destination / "per_image.csv", rows)\n    summary = {"model": args.model, "kind": args.kind, "modality": args.modality,\n               "radiometry": args.radiometry, "trained_on": args.trained_on,\n               "tested_on": domain, "count": len(rows), "parameters": spec["parameters"],\n               "checkpoint_sha256": checkpoint_hash,\n               **{key: float(np.mean([r[key] for r in rows])) for key in rows[0]\n                  if isinstance(rows[0][key], (int, float))}}\n    write_json(result, summary)\n    print(json.dumps(summary, indent=2), flush=True)\n    return summary\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("audit", "train", "evaluate"))\n    parser.add_argument("--oli", type=Path, required=True)\n    parser.add_argument("--mahimandal", type=Path)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--sources", type=Path)\n    parser.add_argument("--kind", choices=("benchmark",), default="benchmark")\n    parser.add_argument("--model", default="bicubic")\n    parser.add_argument("--architecture", type=Path)\n    parser.add_argument("--width", type=int, default=48)\n    parser.add_argument("--modality", choices=tuple(PROFILES), default="rgb")\n    parser.add_argument("--radiometry", choices=("reflectance16", "quantized8"), default="reflectance16")\n    parser.add_argument("--trained-on", choices=("mahimandal", "oli2msi"), default="mahimandal")\n    parser.add_argument("--tested-on", choices=("mahimandal", "oli2msi"), default="oli2msi")\n    parser.add_argument("--checkpoint", type=Path)\n    parser.add_argument("--epochs", type=int, default=50)\n    parser.add_argument("--patience", type=int, default=3)\n    parser.add_argument("--batch-size", type=int, default=1)\n    parser.add_argument("--lr", type=float, default=1e-4)\n    parser.add_argument("--workers", type=int, default=2)\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--border-crop", type=int, default=3)\n    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")\n    args = parser.parse_args()\n    inventory = oli_inventory(args.oli, args.seed)\n    if args.action == "audit":\n        result = {key: value for key, value in inventory.items() if key != "rows"}\n        result["splits"] = {name: len(oli_split(inventory, name)) for name in ("train", "val", "test")}\n        result["geometry"] = [160, 480]\n        write_json(args.output / "oli_audit.json", result)\n        print(json.dumps(result, indent=2), flush=True)\n        return\n    if args.action == "train":\n        print(train(args, inventory))\n        return\n    if args.tested_on == "mahimandal" and args.mahimandal is None:\n        parser.error("--mahimandal is required for that test domain")\n    checkpoint = args.checkpoint or training_root(args) / "best.pt"\n    evaluate(args, inventory, checkpoint, args.tested_on)\n\n\nif __name__ == "__main__":\n    main()\n'}
for name, content in PAYLOAD.items():
    path = owned(CODE_DIR / name)
    if path.exists() and path.read_text(encoding='utf-8') != content:
        raise RuntimeError('Runner snapshot changed; choose a fresh CODE_DIR instead of overwriting an active job')
    if not path.exists(): path.write_text(content, encoding='utf-8')
RUNNER = CODE_DIR / 'cross_dataset_models.py'
registry = json.loads(PAYLOAD['registry.json'])
preflight = json.loads(PREFLIGHT.read_text())
MODELS = sorted({key.split('/')[1] for key, value in preflight.items()
                 if key.startswith('rgb/') and value.get('status') == 'passed'
                 and key.split('/')[1] in registry and 'rgb' in registry[key.split('/')[1]]['modalities']})
if not MODELS: raise RuntimeError('No RGB models passed the original Mahimandal preflight')
BENCH_SOURCES = owned(SOURCE_PATH) if SOURCE_PATH else owned(CROSS_HOME / 'official_sources/notebook4')
BENCH_SOURCES.mkdir(parents=True, exist_ok=True)
sources = {registry[name]['source_dir']: registry[name] for name in MODELS}
for spec in sources.values():
    checkout = owned(BENCH_SOURCES / spec['source_dir'])
    if not checkout.exists():
        run(['git','clone','https://github.com/' + spec['repo'] + '.git',checkout])
    changes = run(['git','-C',checkout,'status','--porcelain','--untracked-files=no'],capture=True).stdout.strip()
    if changes: raise RuntimeError('Preserving modified upstream checkout: ' + str(checkout))
    revision = run(['git','-C',checkout,'rev-parse','HEAD'],capture=True).stdout.strip()
    if revision != spec['revision']:
        if SOURCE_PATH:
            raise RuntimeError('Existing SOURCE_PATH has a different revision; leave SOURCE_PATH empty for isolated sources')
        run(['git','-C',checkout,'fetch','origin',spec['revision']])
        run(['git','-C',checkout,'checkout','--detach',spec['revision']])
COMMON = [PYTHON, RUNNER, '--oli', OLI_ROOT, '--mahimandal', MAH_ROOT,
          '--output', RESULTS, '--sources', BENCH_SOURCES, '--device', DEVICE,
          '--border-crop', 3, '--kind', 'benchmark', '--modality', 'rgb',
          '--radiometry', 'reflectance16', '--trained-on', 'oli2msi', '--seed', SEED]
run(COMMON + ['audit'])
audit = json.loads((RESULTS / 'oli_audit.json').read_text())
if audit['counts'] != {'official_train':5225,'official_test':100}:
    raise ValueError('Unexpected OLI2MSI official counts: ' + str(audit['counts']))
split_code = """import sys
import pandas as pd
from pathlib import Path
from cross_dataset_models import oli_inventory, oli_split
inventory = oli_inventory(Path(sys.argv[1]), int(sys.argv[3]))
pd.DataFrame([{'pair_id':r[0],'split':s} for s in ('train','val','test') for r in oli_split(inventory,s)]).to_csv(sys.argv[2],index=False)
"""
run([PYTHON,'-c',split_code,OLI_ROOT,RESULTS / 'oli_split_manifest.csv',SEED])
print('Models:', MODELS)
print('OLI split counts:', audit['splits'])
print('Training/validation/test: full 160x160 -> 480x480; patience:', PATIENCE)


## Train / resume (validation determines the checkpoint)

In [ ]:
if RUN_TRAINING:
    for model in MODELS:
        print('Training or resuming:', model, flush=True)
        run(COMMON + ['train','--model',model,'--epochs',EPOCHS,'--batch-size',BATCH_SIZE,
                      '--patience',PATIENCE,'--lr',LEARNING_RATE,'--workers',WORKERS])
else:
    print('Training disabled. Existing OLI-trained checkpoints will be used if available.')


## Locked test: both datasets

In [ ]:
if not RUN_TEST:
    raise RuntimeError("Set RUN_TEST=True after validation choices are fixed.")
if not RUN_TEST:
    print('Test remains locked. Freeze validation-selected checkpoints, then set RUN_TEST=True and rerun this cell.')
else:
    for domain in ('oli2msi','mahimandal'):
        for model in ['bicubic'] + MODELS:
            command = COMMON + ['evaluate','--model',model,'--tested-on',domain]
            if model != 'bicubic':
                checkpoint = owned(RESULTS / 'trained_oli2msi/benchmark/reflectance16/rgb' / model / ('seed_' + str(SEED)) / 'best.pt')
                if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)
                command += ['--checkpoint',checkpoint]
            run(command)


## Reports and complete compact download

In [ ]:
import pandas as pd
summary_paths = sorted(RESULTS.glob('evaluation/oli2msi/*/benchmark/reflectance16/rgb/*/*/summary.json'))
if not summary_paths:
    print('No test results yet; run the locked-test cell before exporting.')
else:
    summaries, frames = [], []
    for path in summary_paths:
        info = json.loads(path.read_text())
        if info['model'] not in ['bicubic'] + MODELS: continue
        domain = info['tested_on']
        if info['count'] != (100 if domain == 'oli2msi' else 731):
            raise ValueError('Incomplete evaluation: ' + str(path))
        frame = pd.read_csv(path.with_name('per_image.csv'))
        if len(frame) != info['count'] or frame.pair_id.duplicated().any():
            raise ValueError('Missing/duplicate evaluation rows: ' + str(path))
        summaries.append(info)
        for key in ('model','tested_on','trained_on'): frame[key] = info[key]
        frames.append(frame)
    table = pd.DataFrame(summaries)
    if table.duplicated(['tested_on','model']).any():
        raise RuntimeError('Multiple checkpoints evaluated per model/domain; do not select the best test checkpoint')
    expected = {(d,m) for d in ('oli2msi','mahimandal') for m in ['bicubic']+MODELS}
    if set(zip(table.tested_on,table.model)) != expected:
        raise RuntimeError('Both complete test domains are required before export')
    combined = pd.concat(frames,ignore_index=True)
    table.to_csv(RESULTS/'summary.csv',index=False)
    combined.to_csv(RESULTS/'all_per_image.csv',index=False)
    metrics = [m for m in ('psnr','ssim','mae','rmse','ergas','uiqi','sam_degrees','edge_f1') if m in combined]
    combined.groupby(['tested_on','model','landscape_class'])[metrics].agg(['mean','std','count']).to_csv(RESULTS/'by_landscape.csv')
    combined.groupby(['tested_on','model','geographic_stratum'])[metrics].agg(['mean','std','count']).to_csv(RESULTS/'by_region.csv')
    display(table.sort_values(['tested_on','psnr'],ascending=[True,False]))
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    archive = owned(CROSS_HOME / ('notebook4_complete_compact_' + stamp + '.zip'))
    with zipfile.ZipFile(archive,'x',zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for parent,prefix in ((RESULTS,'results'),(CODE_DIR,'code')):
            for path in sorted(parent.rglob('*')):
                if path.is_file() and '__pycache__' not in path.parts:
                    owned(path)
                    z.write(path,prefix+'/'+path.relative_to(parent).as_posix())
        z.write(PREFLIGHT,'inputs/mahimandal_preflight_status.json')
    print('Download (includes best.pt/last.pt, histories, both domains, figures and code):', archive)
    display(FileLink(os.path.relpath(archive,CWD)))
